In [63]:
#import dask.dataframe as dd
import pandas as pd
from sklearn.preprocessing import StandardScaler,OneHotEncoder
df = pd.read_csv("/Users/hamzatriki/3ºProyectoDeDatos2/PD2/data/ex1/eventos_espera_semana_nuevo.csv", delimiter=",")
df

,ICAO,ultimo_parado,despegue,tiempo_espera,aircraft_type,lat,lon,fecha_despegue,hora_despegue,runway
0,3C5434,2024-12-03 08:17:54.070,2024-12-03 08:28:40.767,646.697,High vortex aircraft,40.496109,-3.574646,2024-12-03,8,18R/36L
1,3C5434,2024-12-03 21:14:30.880,2024-12-03 21:14:49.599,18.719,High vortex aircraft,40.504939,-3.559227,2024-12-03,21,18L/36R
2,44046D,2024-12-01 19:13:45.545,2024-12-01 19:31:45.660,1080.115,High vortex aircraft,40.505637,-3.559243,2024-12-01,19,18L/36R
3,4952CE,2024-12-01 06:55:32.575,2024-12-01 06:55:53.525,20.950,High vortex aircraft,40.496990,-3.574631,2024-12-01,6,18R/36L
4,E8043B,2024-12-02 23:45:55.121,2024-12-02 23:46:35.072,39.951,High vortex aircraft,40.497803,-3.574615,2024-12-02,23,18R/36L
...,...,...,...,...,...,...,...,...,...,...
4033,4D24C4,2024-12-01 20:13:19.998,2024-12-07 17:47:15.489,509635.491,High vortex aircraft,40.469490,-3.566833,2024-12-07,17,14R/32L
4034,4D24C4,2024-12-07 20:13:29.428,2024-12-07 20:15:19.335,109.907,High vortex aircraft,40.505428,-3.559227,2024-12-07,20,18L/36R
4035,A07176,2024-12-02 10:08:36.593,2024-12-02 10:10:56.205,139.612,High vortex aircraft,40.481152,-3.572220,2024-12-02,10,14R/32L
4036,A07176,2024-12-05 23:06:11.578,2024-12-05 23:06:50.548,38.970,High vortex aircraft,40.497828,-3.574630,2024-12-05,23,18R/36L


In [64]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4038 entries, 0 to 4037
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ICAO            4038 non-null   object 
 1   ultimo_parado   4038 non-null   object 
 2   despegue        4038 non-null   object 
 3   tiempo_espera   4038 non-null   float64
 4   aircraft_type   4038 non-null   object 
 5   lat             4038 non-null   float64
 6   lon             4038 non-null   float64
 7   fecha_despegue  4038 non-null   object 
 8   hora_despegue   4038 non-null   int64  
 9   runway          3808 non-null   object 
dtypes: float64(3), int64(1), object(6)
memory usage: 315.6+ KB


In [54]:
icaos = df["ICAO"]
df = df.drop(columns = "ICAO")
df.head()

,ultimo_parado,despegue,tiempo_espera,aircraft_type,lat,lon,fecha_despegue,hora_despegue,runway
0,2024-12-03 08:17:54.070,2024-12-03 08:28:40.767,646.697,High vortex aircraft,40.496109,-3.574646,2024-12-03,8,18R/36L
1,2024-12-03 21:14:30.880,2024-12-03 21:14:49.599,18.719,High vortex aircraft,40.504939,-3.559227,2024-12-03,21,18L/36R
2,2024-12-01 19:13:45.545,2024-12-01 19:31:45.660,1080.115,High vortex aircraft,40.505637,-3.559243,2024-12-01,19,18L/36R
3,2024-12-01 06:55:32.575,2024-12-01 06:55:53.525,20.950,High vortex aircraft,40.496990,-3.574631,2024-12-01,6,18R/36L
4,2024-12-02 23:45:55.121,2024-12-02 23:46:35.072,39.951,High vortex aircraft,40.497803,-3.574615,2024-12-02,23,18R/36L


In [55]:
df.dtypes

ultimo_parado      object
despegue           object
tiempo_espera     float64
aircraft_type      object
lat               float64
lon               float64
fecha_despegue     object
hora_despegue       int64
runway             object
dtype: object

In [56]:
## Cargar bien los tiempos
timestap_columns = ["ultimo_parado","despegue"]
for col in timestap_columns:
    df[col] = pd.to_datetime(df[col],format="%Y-%m-%d %H:%M:%S.%f")
df["fecha_despegue"] = pd.to_datetime(df["fecha_despegue"],format="%Y-%m-%d")
df.dtypes

ultimo_parado     datetime64[ns]
despegue          datetime64[ns]
tiempo_espera            float64
aircraft_type             object
lat                      float64
lon                      float64
fecha_despegue    datetime64[ns]
hora_despegue              int64
runway                    object
dtype: object

In [57]:
categorical_columns = df.select_dtypes("object")
categorical_columns

,aircraft_type,runway
0,High vortex aircraft,18R/36L
1,High vortex aircraft,18L/36R
2,High vortex aircraft,18L/36R
3,High vortex aircraft,18R/36L
4,High vortex aircraft,18R/36L
...,...,...
4033,High vortex aircraft,14R/32L
4034,High vortex aircraft,18L/36R
4035,High vortex aircraft,14R/32L
4036,High vortex aircraft,18R/36L


In [58]:
encoder = OneHotEncoder(sparse_output=False)

categorical_columns = df.select_dtypes("object").columns

encoded_data = encoder.fit_transform(df[categorical_columns])

df_encoded = pd.DataFrame(encoded_data, columns = encoder.get_feature_names_out())
df = df.drop(columns = categorical_columns)
df_encoded = pd.concat([df,df_encoded],axis = 1)

In [59]:
## ultimo parado es el ultimo momento en el que el avion está parado justo antes de despegar, si es 0 ha volado directamente, si es 20segundos hea estadp epsernado 20 segundos en la pista, y despegue cuanto ya esta en el aire,  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4038 entries, 0 to 4037
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ultimo_parado   4038 non-null   datetime64[ns]
 1   despegue        4038 non-null   datetime64[ns]
 2   tiempo_espera   4038 non-null   float64       
 3   lat             4038 non-null   float64       
 4   lon             4038 non-null   float64       
 5   fecha_despegue  4038 non-null   datetime64[ns]
 6   hora_despegue   4038 non-null   int64         
dtypes: datetime64[ns](3), float64(3), int64(1)
memory usage: 221.0 KB


In [61]:
df_encoded.sort_values("ultimo_parado")

,ultimo_parado,despegue,tiempo_espera,lat,lon,fecha_despegue,hora_despegue,aircraft_type_Heavy (larger than 136000 kg),aircraft_type_High vortex aircraft,aircraft_type_Rotorcraft,runway_14L/32R,runway_14R/32L,runway_18L/36R,runway_18R/36L,runway_nan
414,2024-12-01 00:28:51.477,2024-12-01 00:28:56.336,4.859,40.499107,-3.591324,2024-12-01,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
415,2024-12-01 00:28:59.071,2024-12-01 00:28:59.879,0.808,40.499107,-3.591324,2024-12-01,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
416,2024-12-01 00:28:59.879,2024-12-01 00:29:00.281,0.402,40.499109,-3.591328,2024-12-01,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
417,2024-12-01 00:29:00.281,2024-12-01 00:29:00.688,0.407,40.499107,-3.591324,2024-12-01,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
418,2024-12-01 00:29:12.527,2024-12-01 00:29:13.341,0.814,40.499107,-3.591324,2024-12-01,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,2024-12-07 23:27:33.174,2024-12-07 23:27:58.554,25.380,40.497491,-3.574645,2024-12-07,23,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2245,2024-12-07 23:42:20.012,2024-12-07 23:50:03.161,463.149,40.497265,-3.574646,2024-12-07,23,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2705,2024-12-07 23:44:04.299,2024-12-07 23:44:31.284,26.985,40.498306,-3.574646,2024-12-07,23,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
205,2024-12-07 23:46:02.551,2024-12-07 23:47:33.248,90.697,40.498318,-3.574661,2024-12-07,23,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
